In [9]:
%pip install pandas numpy matplotlib seaborn plotly jupyter sqlalchemy


   ------------------------------ --------- 3/4 [isoduration]
   ---------------------------------------- 4/4 [isoduration]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import numpy as np

# Load all tables
orders = pd.read_excel( r"D:\Data Analytics Projects\E-Commerce 360\data\raw\olist_orders_dataset.xlsx", parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items = pd.read_excel(r"D:\Data Analytics Projects\E-Commerce 360\data\raw\olist_order_items_dataset.xlsx")
customers = pd.read_excel(r"D:\Data Analytics Projects\E-Commerce 360\data\raw\olist_customers_dataset.xlsx")
products = pd.read_excel(r"D:\Data Analytics Projects\E-Commerce 360\data\raw\olist_products_dataset.xlsx")
reviews = pd.read_excel(r"D:\Data Analytics Projects\E-Commerce 360\data\raw\olist_order_reviews_dataset.xlsx")
payments = pd.read_excel(r"D:\Data Analytics Projects\E-Commerce 360\data\raw\olist_order_payments_dataset.xlsx")

# --- Data Quality Assessment ---
print("Missing values per column:")
print(orders.isnull().sum())

print("\nDuplicate rows:", orders.duplicated().sum())
print("Order status distribution:")
print(orders['order_status'].value_counts(normalize=True))

# --- Cleaning Steps ---
# Filter to delivered orders only for revenue analysis
delivered = orders[orders['order_status'] == 'delivered'].copy()

# Calculate delivery time in days
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] -
    delivered['order_purchase_timestamp']
).dt.days

# Remove extreme outliers (delivery > 180 days)
delivered = delivered[delivered['delivery_days'].between(0, 180)]

# Create master table with joins
master = (delivered
    .merge(items, on='order_id')
    .merge(customers, on='customer_id')
    .merge(products, on='product_id', how='left')
    .merge(reviews[['order_id','review_score']], on='order_id', how='left')
)

master['total_value'] = master['price'] + master['freight_value']
master['order_month'] = master['order_purchase_timestamp'].dt.to_period('M')
master['order_year'] = master['order_purchase_timestamp'].dt.year

master.to_csv(r"D:\Data Analytics Projects\E-Commerce 360\data\processed\master_orders.csv", index=False)
print(f"Master dataset: {master.shape[0]:,} rows × {master.shape[1]} columns")

Missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate rows: 0
Order status distribution:
order_status
delivered      0.970203
shipped        0.011132
canceled       0.006285
unavailable    0.006124
invoiced       0.003158
processing     0.003027
created        0.000050
approved       0.000020
Name: proportion, dtype: float64
Master dataset: 110,816 rows × 31 columns
